In [1]:
import pickle
from random import randint
import ipywidgets as widgets
from IPython.display import display, Markdown

# Sensible ranges derived from analysing our 6 source blogs' RSS feeds (10th-90th percentile),
# used below instead of hard-coded values when generating text.
from vslm.language_model import (
    MIN_SENTENCES_PER_PARAGRAPH,
    MAX_SENTENCES_PER_PARAGRAPH,
    generate_sentence_forward1,
    generate_sentence_backward1,
    generate_sentence_forward2,
    generate_sentence_backward2,
    generate_sentence_forward3,
    generate_sentence_backward3,
    generate_trigram_sentence as _generate_trigram_sentence,
    least_likely_trigram,
)

In [2]:
forward1 = pickle.load(open("models/forward_1.pkl", "rb"))
backward1 = pickle.load(open("models/backward_1.pkl", "rb"))

# Generate a random sentence.
prompt = 'peatland'
generated_start = list(reversed(generate_sentence_backward1(backward1, prompt)))
generated_end = generate_sentence_forward1(forward1, prompt)
generated_sentence = ' '.join(generated_start)+ ' '+prompt+' '+' '.join(generated_end)+ '.'

out = widgets.Output(layout={'padding': '16px'})
with out:
    display(Markdown('### ' + generated_sentence.capitalize()))
out

Output(layout=Layout(padding='16px'))

In [3]:
forward2 = pickle.load(open("models/forward_2.pkl", "rb"))
backward2 = pickle.load(open("models/backward_2.pkl", "rb"))

# Generate a random sentence using the bigram models, bootstrapped by a one word forecast.
prompt = 'peatland'
generated_start = list(reversed(generate_sentence_backward2(backward2, backward1, prompt)))
generated_end = generate_sentence_forward2(forward2, forward1, prompt)
generated_sentence = ' '.join(generated_start)+ ' '+prompt+' '+' '.join(generated_end)+ '.'

out = widgets.Output(layout={'padding': '16px'})
with out:
    display(Markdown('### ' + generated_sentence.capitalize()))
out

Output(layout=Layout(padding='16px'))

In [4]:
forward3 = pickle.load(open("models/forward_3.pkl", "rb"))
backward3 = pickle.load(open("models/backward_3.pkl", "rb"))

# Generate a random sentence using the trigram models, bootstrapped by unigram and bigram forecasts.
prompt = 'peatland'
generated_start = list(reversed(generate_sentence_backward3(backward3, backward2, backward1, prompt)))
generated_end = generate_sentence_forward3(forward3, forward2, forward1, prompt)
generated_sentence = ' '.join(generated_start)+ ' '+prompt+' '+' '.join(generated_end)+ '.'

out = widgets.Output(layout={'padding': '16px'})
with out:
    display(Markdown('### ' + generated_sentence.capitalize()))
out

Output(layout=Layout(padding='16px'))

In [5]:
words3 = pickle.load(open("models/words_3.pkl", "rb"))

def generate_trigram_sentence(seed_word):
    # Build one full sentence (as a list of words) using the trigram models.
    return _generate_trigram_sentence(
        seed_word, forward3, forward2, forward1, backward3, backward2, backward1
    )

def next_seed_word(sentence_words, prompt):
    # Pick one of the three words from the least likely (but still pre-occurring) trigram,
    # or the prompt, to seed the next sentence. (least_likely_trigram is imported above from
    # vslm/language_model.py, but this function itself is duplicated locally here.)
    triple = least_likely_trigram(sentence_words, words3)
    if triple is None:
        return prompt
    choices = [triple[0], triple[1], triple[2], prompt]
    return choices[randint(0, 3)]


# Generate a paragraph: each sentence is built with the trigram method, then the least
# statistically likely (but still pre-occurring) trigram in it is found and one of its three
# words (or the prompt) seeds the next sentence. The number of sentences is sampled from the
# sensible sentences-per-paragraph range observed in our sources (roughly 1-4 sentences).
prompt = 'peatland'
seed_word = prompt
paragraph_sentences = []
for _ in range(randint(MIN_SENTENCES_PER_PARAGRAPH, MAX_SENTENCES_PER_PARAGRAPH)):
    sentence_words = generate_trigram_sentence(seed_word)
    sentence_text = ' '.join(sentence_words) + '.'
    paragraph_sentences.append(sentence_text.capitalize())
    seed_word = next_seed_word(sentence_words, seed_word)
paragraph = ' '.join(paragraph_sentences)

out = widgets.Output(layout={'padding': '16px'})
with out:
    display(Markdown('### ' + paragraph))
out

Output(layout=Layout(padding='16px'))